# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [28]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print("Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

Rozpoczynam pobieranie SEKWENCYJNE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Dziwny przypadek psa nocną porą
2. Koncert oratoryjno-pasyjny AMKP
3. Międzynarodowy Dzień Poezji z Krakowem Miastem Literatury UNESCO
4. Śpiewoterapia
5. Czytanie na dywanie
6. Alicja w Krainie Czarów
7. Impro KRK Underground
8. Jestem obok. Wszyscy w domu
9. Bal
10. Amirova Trio & Iwona Karcz-Wojnarowska: Kiedy tradycja spotyka jazz

Czas wykonania: 6.64s


In [2]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print("Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Dziwny przypadek psa nocną porą
2. Koncert oratoryjno-pasyjny AMKP
3. Międzynarodowy Dzień Poezji z Krakowem Miastem Literatury UNESCO
4. Śpiewoterapia
5. Czytanie na dywanie
6. Alicja w Krainie Czarów
7. Impro KRK Underground
8. Jestem obok. Wszyscy w domu
9. Bal
10. Amirova Trio & Iwona Karcz-Wojnarowska: Kiedy tradycja spotyka jazz

Czas wykonania (wątki): 1.77s


--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [5]:
import threading
import concurrent

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

Saldo końcowe: 100 zł (oczekiwano: 100)


--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [3]:
import multiprocessing
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

Praca na 8 procesach (rdzeniach)...
Zakończono w czasie 1.48s.


---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [31]:
import concurrent.futures
import requests
import time

fact_count = 20
api_url = "https://catfact.ninja/fact"

def fetch_cat_fact(url=api_url, timeout=10):
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()
    return response.json().get("fact")

def fetch_facts_sequential(count=fact_count):
    start = time.time()
    facts = [fetch_cat_fact() for _ in range(count)]
    duration = time.time() - start
    return facts, duration

def fetch_facts_threaded(count=fact_count, max_workers=fact_count):
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        facts = []
        tasks = [executor.submit(fetch_cat_fact) for _ in range(count)]
        for t in tasks:
            facts.append(t.result())

    duration = time.time() - start
    return facts, duration

sequntial_facts, sequential_time = fetch_facts_sequential()
threaded_facts, threaded_time = fetch_facts_threaded()

print(f"Sekwencyjnie pobrano {len(sequntial_facts)} faktów w {sequential_time:.2f}s")
print(f"Wielowątkowo pobrano {len(threaded_facts)} faktów w {threaded_time:.2f}s")
if threaded_time > 0:
    print(f"Przyspieszenie: {sequential_time / threaded_time:.2f}x")

for i in range(len(threaded_facts)):
    print(f"Fakt {i+1}: {threaded_facts[i]}")

Sekwencyjnie pobrano 20 faktów w 16.33s
Wielowątkowo pobrano 20 faktów w 3.15s
Przyspieszenie: 5.18x
Fakt 1: A cat's appetite is the barometer of its health. Any cat that does not eat or drink for more than two days should be taken to a vet.
Fakt 2: During the time of the Spanish Inquisition, Pope Innocent VIII condemned cats as evil and thousands of cats were burned. Unfortunately, the widespread killing of cats led to an explosion of the rat population, which exacerbated the effects of the Black Death.
Fakt 3: The chlorine in fresh tap water irritates sensitive parts of the cat's nose. Let tap water sit for 24 hours before giving it to a cat.
Fakt 4: The life expectancy of cats has nearly doubled over the last fifty years.
Fakt 5: The normal body temperature of a cat is between 100.5 ° and 102.5 °F. A cat is sick if its temperature goes below 100 ° or above 103 °F.
Fakt 6: A cat has two vocal chords, and can make over 100 sounds.
Fakt 7: The largest breed of cat is the Ragdoll with m

### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [ ]:
import queue
import threading
import time

def producer(q: queue.Queue, max_number: int):
    for i in range(1, max_number + 1):
        q.put(i)
        time.sleep(0.1) 

    q.put(None)
    q.put(None)

def consumer_even(q: queue.Queue, consumer_id: int):
    even_numbers = []
    while True:
        number = q.get()
        if number is None:
            print(f"Konsument {consumer_id} przetworzył: {even_numbers}")
            break
        if number % 2 == 0:
            even_numbers.append(number)
        else:
            q.put(number)

def consumer_odd(q: queue.Queue, consumer_id: int):
    odd_numbers = []
    while True:
        number = q.get()
        if number is None:
            print(f"Konsument {consumer_id} przetworzył: {odd_numbers}")
            break
        if number % 2 == 1:
            odd_numbers.append(number)
        else:
            q.put(number)

# Start
if __name__ == "__main__":
    
    q = queue.Queue()    
    t_producer = threading.Thread(target=producer, args=(q, 20))
    t_consumer_even = threading.Thread(target=consumer_even, args=(q, 1))
    t_consumer_odd = threading.Thread(target=consumer_odd, args=(q, 2))

    t_producer.start()
    t_consumer_even.start()
    t_consumer_odd.start()

    t_producer.join()
    t_consumer_even.join()
    t_consumer_odd.join()

Konsument 1 przetworzył: [2, 4, 6, 8, 10, 12, 14, 16, 18, 20]
Konsument 2 przetworzył: [1, 3, 5, 7, 9, 11, 13, 15, 17, 19]


### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [14]:
import multiprocessing
import time
from lab2_functions import calculate_power_sum

if __name__ == "__main__":
    max_number = 10000
    
    print(f"Obliczanie sumy potęg dla liczb 1-{max_number}...")
    print(f"Liczba procesów: {multiprocessing.cpu_count()}\n")
    
    start = time.time()
    
    with multiprocessing.Pool(processes=multiprocessing.cpu_count()) as pool:
        results = pool.map(calculate_power_sum, range(1, max_number + 1))
    
    elapsed = time.time() - start
    
    print(f"Obliczono {len(results)} sum potęg")
    print(f"Przykłady wyników:")
    for i in [0, 99, 999, 9999]:
        print(f"  calculate_power_sum({i+1}) = {results[i]}")
    
    total_sum = sum(results)
    print(f"\nSuma wszystkich wyników: {total_sum}")
    print(f"Czas wykonania: {elapsed:.2f}s")

Obliczanie sumy potęg dla liczb 1-10000...
Liczba procesów: 4

Obliczono 10000 sum potęg
Przykłady wyników:
  calculate_power_sum(1) = 100
  calculate_power_sum(100) = 101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010100
  calculate_power_sum(1000) = 1001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001001000
  calculate_power_sum(10000) = 100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100010001000100